In [2]:
import os
from pathlib import Path
from tqdm.auto import tqdm
import hashlib
from io import BytesIO

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from datasets import load_dataset
from transformers import T5TokenizerFast
import requests
from urllib import request as urllib_request
import random
from collections import defaultdict
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import T5TokenizerFast
from tqdm.auto import tqdm




In [3]:
DATA_DIR = Path("./kvasir_vqa_subset")
IMG_DIR = DATA_DIR / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

HF_DATASET_ID = "SimulaMet/Kvasir-VQA-x1"   
SPLIT = "train"                            
N_IMAGES = 20000                           # number of unique images to save
EARLY_ROWS_MARGIN = 0.10                   # stop after N_IMAGES and N*(1+margin) QA rows collected
MAX_ROWS_SCAN = 250000                     # safety cap on rows scanned
PRINT_EVERY_N_IMAGES = 1000    

TOKENIZER_NAME = "t5-small"
IMG_SIZE = 224
BATCH_SIZE = 6            # start small on 8GB GPU; increase if memory allows
NUM_WORKERS = 4
VAL_IMAGE_SPLIT = 0.05    # fraction of unique images used for validation
IMAGE_LEVEL_SPLIT = True  # True => ensure no image leakage between train/val
FILTER_COMPLEXITIES = None  # e.g. [1,2] or None to keep all
MAX_QA_PER_IMAGE = None   # e.g. 5 to limit QA per image (None => keep all)
SHUFFLE_SEED = 42
PREFIX = "<image> "       # prefix inserted before question text
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data Loader Code 

In [ ]:


CSV_PATH = "kvasir_vqa_subset/kvasir_vqa_subset_info_normalized.csv"
IMG_DIR = Path("kvasir_vqa_subset/images")

NUM_WORKERS = 0      # IMPORTANT: avoid multiprocessing hangs in Jupyter on Windows
BATCH_SIZE = 16       # small for quick check
IMG_SIZE = 224
VAL_IMAGE_SPLIT = 0.05
TOKENIZER_NAME = "t5-small"
PREFIX = "<image> "

# Load normalized CSV
df = pd.read_csv(CSV_PATH)
# If you have 'resolved_img_path' keep it; otherwise construct from img_id
if "resolved_img_path" in df.columns:
    df["img_path"] = df["resolved_img_path"]
elif "img_path" not in df.columns and "img_id" in df.columns:
    df["img_path"] = df["img_id"].astype(str).apply(lambda x: str(IMG_DIR / f"{x}.jpg"))

# drop rows with missing image files (defensive)
from pathlib import Path
df["img_path"] = df["img_path"].astype(str)
df = df[df["img_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
print("Rows after resolving image paths:", len(df))


if "split" in df.columns:
    df_train = df[df["split"] == "train"].reset_index(drop=True)
    df_test  = df[df["split"] == "test"].reset_index(drop=True)
    print("Using 'split' column in CSV: train rows:", len(df_train), "test rows:", len(df_test))
else:
    # image-level split to create a validation set (no leakage)
    grouped = defaultdict(list)
    for _, r in df.iterrows():
        grouped[r["img_id"]].append(r.to_dict())
    img_ids = list(grouped.keys())
    random.seed(42)
    random.shuffle(img_ids)
    num_val = max(1, int(len(img_ids) * VAL_IMAGE_SPLIT))
    val_ids = set(img_ids[:num_val])
    train_ids = set(img_ids[num_val:])
    train_rows = [row for img in train_ids for row in grouped[img]]
    val_rows   = [row for img in val_ids for row in grouped[img]]
    # for convenience, treat val_rows as test set here
    df_train = pd.DataFrame(train_rows)
    df_test  = pd.DataFrame(val_rows)
    print("Created train/test splits by image. Train rows:", len(df_train), "Test rows:", len(df_test))

# now create rows list used by Dataset
def rows_from_df(dfin):
    rows = []
    for _, r in dfin.iterrows():
        rows.append({
            "img_path": r["img_path"],
            "img_id": str(r["img_id"]),
            "question": (r.get("question") or "").strip(),
            "answer": (r.get("answer") or "").strip()
        })
    return rows

train_rows = rows_from_df(df_train)
test_rows  = rows_from_df(df_test)

print(f"Train images (unique): {len(set(r['img_id'] for r in train_rows))}, Train rows: {len(train_rows)}")
print(f"Test images  (unique): {len(set(r['img_id'] for r in test_rows))}, Test rows: {len(test_rows)}")

# tokenizer + transforms
tokenizer = T5TokenizerFast.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class KvasirVQADataset(Dataset):
    def __init__(self, rows, tokenizer, transforms, max_q_len=64, max_a_len=64, prefix=PREFIX):
        self.rows = rows
        self.tokenizer = tokenizer
        self.transforms = transforms
        self.max_q_len = max_q_len
        self.max_a_len = max_a_len
        self.prefix = prefix
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        r = self.rows[idx]
        img = Image.open(r["img_path"]).convert("RGB")
        img = self.transforms(img)
        q = f"{self.prefix}{r['question']}"
        q_tok = self.tokenizer(q, truncation=True, max_length=self.max_q_len, padding=False)
        a_tok = self.tokenizer(r['answer'], truncation=True, max_length=self.max_a_len, padding=False)
        return {
            "pixel_values": img,
            "input_ids": torch.tensor(q_tok["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(q_tok["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(a_tok["input_ids"], dtype=torch.long),
            "img_id": r["img_id"]
        }

def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch], dim=0)
    input_ids = [b["input_ids"] for b in batch]
    attention_mask = [b["attention_mask"] for b in batch]
    labels = [b["labels"] for b in batch]
    tok_batch = tokenizer.pad({"input_ids": input_ids, "attention_mask": attention_mask},
                              padding="longest", return_tensors="pt")
    labels_batch = tokenizer.pad({"input_ids": labels}, padding="longest", return_tensors="pt")["input_ids"]
    labels_batch[labels_batch == tokenizer.pad_token_id] = -100
    return {
        "pixel_values": pixel_values,
        "input_ids": tok_batch["input_ids"],
        "attention_mask": tok_batch["attention_mask"],
        "labels": labels_batch,
        "img_ids": [b["img_id"] for b in batch]
    }

train_ds = KvasirVQADataset(train_rows, tokenizer, train_transforms)
val_ds   = KvasirVQADataset(test_rows, tokenizer, val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print("Dataloaders created. Train batches:", len(train_loader), "Val/Test batches:", len(val_loader))

# Quick sanity-check (fetch a single batch)
batch = next(iter(train_loader))
print("Sample batch shapes:", batch["pixel_values"].shape, batch["input_ids"].shape, batch["labels"].shape)
print("Decoded Q:", tokenizer.decode(batch["input_ids"][0], skip_special_tokens=True))
print("Decoded A:", tokenizer.decode(batch["labels"][0].masked_fill(batch["labels"][0]==-100, tokenizer.pad_token_id), skip_special_tokens=True))


Rows after resolving image paths: 159549
Using 'split' column in CSV: train rows: 143594 test rows: 15955
Train images (unique): 6212, Train rows: 143594
Test images  (unique): 4058, Test rows: 15955


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Dataloaders created. Train batches: 8975 Val/Test batches: 998
Sample batch shapes: torch.Size([16, 3, 224, 224]) torch.Size([16, 39]) torch.Size([16, 31])
Decoded Q: image> In which regions of the image is the abnormality located?
Decoded A: abnormality distributed across multiple quadrants


# Main architecture of the project  begains  here

In [ ]:


import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, Dict
from transformers import T5ForConditionalGeneration, T5TokenizerFast
from tqdm.auto import tqdm

try:
    from torchvision.models import resnet18, ResNet18_Weights
    def make_resnet18(pretrained=True):
        if pretrained:
            try:
                return resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
            except Exception:
                return resnet18(pretrained=True)
        else:
            return resnet18(weights=None)
except Exception:
    from torchvision.models import resnet18
    def make_resnet18(pretrained=True):
        return resnet18(pretrained=pretrained)

class ImageEncoderResNet18(nn.Module):
    """ResNet18 conv backbone returning pooled vector (B,C) or spatial tokens (B,V,C)."""
    def __init__(self, pretrained: bool = True, spatial: bool = True):
        super().__init__()
        self.spatial = spatial
        base = make_resnet18(pretrained=pretrained)
        modules = list(base.children())[:-2]  
        self.features = nn.Sequential(*modules)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.out_dim = 512
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.features(x)             
        B, C, H, W = feats.shape
        if not self.spatial:
            pooled = self.pool(feats).view(B, C)
            return pooled
        tokens = feats.view(B, C, H*W).permute(0, 2, 1).contiguous()  
        return tokens

class QFormer(nn.Module):
    """
    Tiny Q-former: learnable M queries cross-attend to image patch tokens.
    Uses multihead attention where query dim = q_dim and keys/vals are projected to q_dim.
    """
    def __init__(self, n_q: int = 4, q_dim: int = 256, kv_dim: int = 512,
                 n_heads: int = 8, ff_dim: int = 1024, dropout: float = 0.1, num_layers: int = 1):
        super().__init__()
        self.n_q = n_q
        self.q_dim = q_dim
        self.kv_dim = kv_dim
        self.num_layers = num_layers

        self.queries = nn.Parameter(torch.randn(n_q, q_dim) * 0.02)

        self.kv_proj = nn.Linear(kv_dim, q_dim) if kv_dim != q_dim else None

        self.cross_attn_layers = nn.ModuleList()
        self.ffns = nn.ModuleList()
        for _ in range(num_layers):
            self.cross_attn_layers.append(
                nn.MultiheadAttention(embed_dim=q_dim, kdim=q_dim, vdim=q_dim,
                                      num_heads=n_heads, dropout=dropout, batch_first=True)
            )
            self.ffns.append(
                nn.Sequential(
                    nn.Linear(q_dim, ff_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(ff_dim, q_dim),
                    nn.Dropout(dropout),
                    nn.LayerNorm(q_dim)
                )
            )
        self.layernorm = nn.LayerNorm(q_dim)

    def forward(self, image_tokens: torch.Tensor) -> torch.Tensor:
        """
        image_tokens: [B, V, C_v]
        returns: q_out [B, M, q_dim]
        """
        B, V, C_v = image_tokens.shape
        q = self.queries.unsqueeze(0).expand(B, -1, -1).contiguous()  
        if self.kv_proj is not None:
            k = self.kv_proj(image_tokens) 
            v = k
        else:
            k = image_tokens
            v = k
        x_q = q
        for attn, ffn in zip(self.cross_attn_layers, self.ffns):
            attn_out, _ = attn(x_q, k, v, need_weights=False)  # [B, M, q_dim]
            x_q = x_q + attn_out
            x_q = x_q + ffn(x_q)
            x_q = self.layernorm(x_q)
        return x_q  # [B, M, q_dim]

class BLIP2_T5(nn.Module):
    """
    BLIP-2-like wiring:
      ResNet18 (frozen) -> spatial tokens -> QFormer -> project -> prepend to T5 encoder embeddings
      T5 encoder is frozen; train Q-former, projector, and T5 decoder.
    """
    def __init__(self, t5_name: str = "t5-small", q_n: int = 4, q_dim: int = 256,
                 q_num_layers: int = 1, freeze_img_encoder: bool = True, freeze_t5_encoder: bool = True,
                 pretrained_img: bool = True):
        super().__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(t5_name)
        self.tokenizer = T5TokenizerFast.from_pretrained(t5_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.add_special_tokens({"pad_token": "<pad>"})
        self.d_model = self.t5.config.d_model

        self.img_encoder = ImageEncoderResNet18(pretrained=pretrained_img, spatial=True)

        self.q_former = QFormer(n_q=q_n, q_dim=q_dim, kv_dim=self.img_encoder.out_dim,
                                num_layers=q_num_layers)

        self.q_to_t5 = nn.Sequential(
            nn.Linear(q_dim, self.d_model),
            nn.LayerNorm(self.d_model),
            nn.Dropout(0.1)
        )

        # learned positional embeddings for image tokens
        self.img_token_pos = nn.Parameter(torch.randn(q_n, self.d_model) * 0.02)

        # freeze flags (recommended for 8GB)
        if freeze_img_encoder:
            for p in self.img_encoder.parameters():
                p.requires_grad = False
        if freeze_t5_encoder:
            for n, p in self.t5.get_encoder().named_parameters():
                p.requires_grad = False

    def encode_image(self, pixel_values: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Convert pixel_values -> img_embeds [B, M, d_model] and pooled bottleneck repr q_repr [B, d_model]
        """
        img_feats = self.img_encoder(pixel_values)   # [B, V, C_v]
        q_out = self.q_former(img_feats)             # [B, M, q_dim]
        img_embeds = self.q_to_t5(q_out)             # [B, M, d_model]
        img_embeds = img_embeds + self.img_token_pos.unsqueeze(0).to(img_embeds.device).to(img_embeds.dtype)
        q_repr = img_embeds.mean(dim=1)              # pooled bottleneck for contrastive
        return img_embeds, q_repr

    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                labels: Optional[torch.Tensor] = None) -> Tuple[object, Dict]:
        B = pixel_values.size(0)
        img_embeds, q_repr = self.encode_image(pixel_values)  # [B, M, d], [B, d]
        # get token embeddings from T5 encoder embed_tokens
        embed_tokens = self.t5.get_encoder().embed_tokens
        text_embeds = embed_tokens(input_ids)  # [B, Lq, d_model]
        # concat and build attention mask
        encoder_inputs_embeds = torch.cat([img_embeds, text_embeds], dim=1)  # [B, M+Lq, d]
        img_mask = torch.ones(B, img_embeds.size(1), dtype=attention_mask.dtype, device=attention_mask.device)
        encoder_attention_mask = torch.cat([img_mask, attention_mask], dim=1)  # [B, M+Lq]
        # run T5 encoder with inputs_embeds
        encoder_outputs = self.t5.get_encoder()(inputs_embeds=encoder_inputs_embeds,
                                                attention_mask=encoder_attention_mask,
                                                return_dict=True)
        outputs = self.t5(encoder_outputs=encoder_outputs,
                          attention_mask=encoder_attention_mask,
                          labels=labels,
                          return_dict=True)
        aux = {"encoder_outputs": encoder_outputs, "q_repr": q_repr, "img_embeds": img_embeds, "text_embeds": text_embeds}
        return outputs, aux

    def generate(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                 max_length: int = 64, num_beams: int = 3, **gen_kwargs):
        img_embeds, _ = self.encode_image(pixel_values)
        text_embeds = self.t5.get_encoder().embed_tokens(input_ids)
        encoder_inputs_embeds = torch.cat([img_embeds, text_embeds], dim=1)
        img_mask = torch.ones(pixel_values.size(0), img_embeds.size(1),
                              dtype=attention_mask.dtype, device=attention_mask.device)
        encoder_attention_mask = torch.cat([img_mask, attention_mask], dim=1)
        encoder_outputs = self.t5.get_encoder()(inputs_embeds=encoder_inputs_embeds,
                                                attention_mask=encoder_attention_mask,
                                                return_dict=True)
        generated_ids = self.t5.generate(encoder_outputs=encoder_outputs,
                                        attention_mask=encoder_attention_mask,
                                        max_length=max_length,
                                        num_beams=num_beams,
                                        **gen_kwargs)
        return generated_ids

# ---- Contrastive loss helper (symmetric InfoNCE) ----
def compute_contrastive_loss(img_repr: torch.Tensor, text_repr: torch.Tensor, temp: float = 0.07) -> torch.Tensor:
    """
    img_repr: [B, d], text_repr: [B, d] (both projected to same d_model)
    returns symmetric InfoNCE scalar loss
    """
    img_n = F.normalize(img_repr, dim=-1)
    txt_n = F.normalize(text_repr, dim=-1)
    logits = (img_n @ txt_n.t()) / temp   # [B, B]
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_i = F.cross_entropy(logits, labels)
    loss_t = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_i + loss_t)

# ---- Instantiate model, optimizer, sample training step (smoke) ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# instantiate model (this will download t5-small if not cached)
model_blip = BLIP2_T5(t5_name="t5-small", q_n=4, q_dim=256, q_num_layers=1,
                      freeze_img_encoder=True, freeze_t5_encoder=True, pretrained_img=True)
model_blip.to(device)

# Which params are trainable?
trainable = [p for p in model_blip.parameters() if p.requires_grad]
print("Trainable param groups:", sum(p.numel() for p in trainable), "total params:", sum(p.numel() for p in model_blip.parameters()))

# Example optimizer: train Q-former, q_to_t5, and T5 decoder (decoder params are trainable while encoder frozen)
optimizer = torch.optim.AdamW(trainable, lr=5e-5, weight_decay=0.01)

# Check train_loader existence and run a single combined step
if "train_loader" not in globals():
    print("train_loader not found. Model defined and ready. Run your dataloader cell and then use the training loop cell I'll provide next.")
else:
    batch = next(iter(train_loader))
    pixel_values = batch["pixel_values"].to(device)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    use_amp = (device == "cuda")
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    model_blip.train()
    with torch.cuda.amp.autocast(enabled=use_amp):
        outputs, aux = model_blip(pixel_values, input_ids, attention_mask, labels=labels)
        ce_loss = outputs.loss  # cross-entropy
        # text pooled repr: mean of token embeddings BEFORE encoder (we used embed_tokens)
        # project text embedding pooled to d_model (they already are d_model since we used embed_tokens)
        text_repr = aux["text_embeds"].mean(dim=1)  # [B, d_model]
        img_repr = aux["q_repr"]                    # [B, d_model]
        contrastive_loss = compute_contrastive_loss(img_repr, text_repr, temp=0.07)
        alpha = 0.1  # contrastive weight (tune: 0.05 - 0.2)
        loss = ce_loss + alpha * contrastive_loss

    # backward + step
    optimizer.zero_grad()
    if use_amp:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    print(f"CE loss: {float(ce_loss.item()):.4f}, contrastive: {float(contrastive_loss.item()):.4f}, total: {float(loss.item()):.4f}")
    # generation smoke test
    model_blip.eval()
    with torch.no_grad():
        gen_ids = model_blip.generate(pixel_values, input_ids, attention_mask, max_length=64, num_beams=3)
        decoded = model_blip.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        print("Sample generated (first 4):")
        for i, s in enumerate(decoded[:4]):
            print(f"[{i}] {s}")

# ---- Notes ----
# - This cell freezes image encoder and T5 encoder to fit 8GB. To fine-tune further: unfreeze small parts with lower LR.
# - Contrastive alpha, q_n (M), q_dim can be tuned. Start with q_n=4, q_dim=256, alpha=0.1.
# - For better alignment use CLIP text embeddings in place of 'text_repr' (optional).
# - Next step: I can give you a full training loop with accumulation, scheduler, and validation using this model (tuned for 8GB).


Device: cuda
Trainable param groups: 26232576 total params: 72739904


C:\Users\sayantan\AppData\Local\Temp\ipykernel_16316\2999614704.py:241: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None
C:\Users\sayantan\AppData\Local\Temp\ipykernel_16316\2999614704.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


CE loss: 3.8035, contrastive: 2.7911, total: 4.0826
Sample generated (first 4):
[0] image> All polyps have been completely removed, what is the size of the remaining polyp, and what type of polyp is present?
[1] image> Haben alle polyps been removed, where are the anatomical landmarks located, and what is the size of any polyps present?
[2] image> Which anatomical landmark is visible in the image and what are the colors of the abnormality?
[3] image> image> image> image> image> image> image> image> image> image> image> image> image> image> image> image


In [ ]:
# === BLIP-2 training loop with contrastive loss, AMP, accumulation, validation, checkpointing ===
import os, math, time
from tqdm.auto import tqdm
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

# ------------- CONFIG -------------
EPOCHS = 5
ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
SAVE_DIR = "checkpoints_blip2_t5"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_GEN_LEN = 64
NUM_BEAMS = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LOG_EVERY_N_STEPS = 50
EVAL_EVERY_EPOCHS = 1

# contrastive weight
ALPHA = 0.1  # try 0.05 - 0.2
CONTRASTIVE_TEMP = 0.07
# ----------------------------------

# Resolve model instance: prefer model_blip, else model_q/model, else instantiate if class is present
if "model_blip" in globals():
    model = globals()["model_blip"]
    print("Using existing model_blip.")
elif "model_q" in globals():
    model = globals()["model_q"]
    print("Using model_q (renamed locally to model).")
elif "model" in globals():
    model = globals()["model"]
    print("Using model.")
else:
    if "BLIP2_T5" in globals():
        print("Instantiating BLIP2_T5 with defaults (q_n=4, q_dim=256).")
        model = globals()["BLIP2_T5"](t5_name="t5-small", q_n=4, q_dim=256, q_num_layers=1,
                                       freeze_img_encoder=True, freeze_t5_encoder=True, pretrained_img=True)
        globals()["model_blip"] = model
    else:
        raise RuntimeError("No model found. Define/instantiate model_blip, model_q or BLIP2_T5 class first.")

# Check dataloaders
if "train_loader" not in globals() or "val_loader" not in globals():
    raise RuntimeError("train_loader or val_loader not found. Run dataloader cells first.")

model.to(DEVICE)

# Trainable params (automatically selects those with requires_grad)
trainable_params = [p for p in model.parameters() if p.requires_grad]
if len(trainable_params) == 0:
    raise RuntimeError("No trainable parameters. Unfreeze Q-former / T5 decoder or check model init.")

optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
max_train_steps = EPOCHS * steps_per_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=max_train_steps)

use_amp = DEVICE == "cuda"
scaler = torch.cuda.amp.GradScaler() if use_amp else None

# helper: symmetric InfoNCE
def symmetric_contrastive_loss(img_repr, txt_repr, temp=CONTRASTIVE_TEMP):
    img_n = F.normalize(img_repr, dim=-1)
    txt_n = F.normalize(txt_repr, dim=-1)
    logits = (img_n @ txt_n.t()) / temp
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_i = F.cross_entropy(logits, labels)
    loss_t = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_i + loss_t)

# checkpoint save
def save_checkpoint(epoch, step, model, optimizer, scheduler, best_metric, path):
    state = {
        "epoch": epoch,
        "step": step,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_metric": best_metric
    }
    torch.save(state, path)

# metrics helpers (simple)
def normalize_text(s: str) -> str:
    return " ".join(str(s).lower().strip().split())

def exact_match(pred, ref):
    return int(normalize_text(pred) == normalize_text(ref))

def rouge_l_score(pred, ref):
    pred_tokens = normalize_text(pred).split()
    ref_tokens = normalize_text(ref).split()
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0
    m, n = len(pred_tokens), len(ref_tokens)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        for j in range(n):
            if pred_tokens[i] == ref_tokens[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    lcs = dp[m][n]
    recall = lcs / n if n > 0 else 0.0
    prec = lcs / m if m > 0 else 0.0
    if recall + prec == 0:
        return 0.0
    beta2 = 1.0
    return (1 + beta2) * prec * recall / (recall + beta2 * prec)

# training loop
best_rouge = -1.0
global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    train_iter = enumerate(train_loader)
    progress = tqdm(train_iter, total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS} - training")
    for step, batch in progress:
        pixel_values = batch["pixel_values"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs, aux = model(pixel_values, input_ids, attention_mask, labels=labels)
            ce_loss = outputs.loss
            # compute contrastive only when effective batch > 1
            bsz = aux["q_repr"].size(0)
            if bsz > 1:
                # text embeddings were stored in aux as token embeddings before encoder
                text_repr = aux["text_embeds"].mean(dim=1)  # [B, d_model]
                img_repr = aux["q_repr"]                    # [B, d_model]
                contrastive_loss = symmetric_contrastive_loss(img_repr, text_repr)
            else:
                contrastive_loss = torch.tensor(0.0, device=ce_loss.device)

            total_loss = ce_loss + ALPHA * contrastive_loss

            # normalize for gradient accumulation
            loss_to_backprop = total_loss / ACCUMULATION_STEPS

        if use_amp:
            scaler.scale(loss_to_backprop).backward()
        else:
            loss_to_backprop.backward()

        epoch_loss += float(total_loss.detach().cpu().item())

        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            if use_amp:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        # logging
        if (step + 1) % LOG_EVERY_N_STEPS == 0:
            avg_loss = epoch_loss / (step + 1)
            lr = scheduler.get_last_lr()[0] if hasattr(scheduler, "get_last_lr") else None
            progress.set_postfix({"avg_loss": f"{avg_loss:.4f}", "lr": lr})

    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch} training finished. avg_loss: {avg_epoch_loss:.4f}")

    # validation
    if epoch % EVAL_EVERY_EPOCHS == 0:
        model.eval()
        val_loss = 0.0
        all_preds, all_refs = [], []
        with torch.no_grad():
            val_progress = tqdm(enumerate(val_loader), total=len(val_loader), desc=f"Epoch {epoch} - validating")
            for vstep, batch in val_progress:
                pixel_values = batch["pixel_values"].to(DEVICE)
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs, aux = model(pixel_values, input_ids, attention_mask, labels=labels)
                    vloss = outputs.loss
                val_loss += float(vloss.detach().cpu().item())

                # generation
                gen_ids = model.generate(pixel_values, input_ids, attention_mask, max_length=MAX_GEN_LEN, num_beams=NUM_BEAMS)
                decoded_preds = model.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

                labels_cpu = labels.clone().detach().cpu()
                labels_cpu[labels_cpu == -100] = model.tokenizer.pad_token_id
                decoded_refs = model.tokenizer.batch_decode(labels_cpu, skip_special_tokens=True)

                all_preds.extend(decoded_preds)
                all_refs.extend(decoded_refs)

        em_count = sum(exact_match(p, r) for p, r in zip(all_preds, all_refs))
        em = em_count / len(all_preds) if all_preds else 0.0
        rouge_l_vals = [rouge_l_score(p, r) for p, r in zip(all_preds, all_refs)]
        rouge_l = sum(rouge_l_vals) / len(rouge_l_vals) if rouge_l_vals else 0.0
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0.0

        print(f"\nValidation -- avg_loss: {avg_val_loss:.4f}, EM: {em:.4f}, ROUGE-L: {rouge_l:.4f}")
        # print some examples
        for i in range(min(3, len(all_preds))):
            print(" PRED:", all_preds[i])
            print(" TRUE:", all_refs[i])
            print("----")

        # save best
        if rouge_l > best_rouge:
            best_rouge = rouge_l
            ckpt_path = os.path.join(SAVE_DIR, f"best_epoch{epoch}_rouge{rouge_l:.4f}.pt")
            save_checkpoint(epoch, global_step, model, optimizer, scheduler, best_rouge, ckpt_path)
            print("Saved best model to", ckpt_path)

print("\nTraining finished. Best ROUGE-L:", best_rouge)
final_path = os.path.join(SAVE_DIR, "final_model.pt")
save_checkpoint(EPOCHS, global_step, model, optimizer, scheduler, best_rouge, final_path)
print("Final model saved to:", final_path)


C:\Users\sayantan\AppData\Local\Temp\ipykernel_16316\1858854298.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


Using existing model_blip.


Epoch 1/5 - training:   0%|          | 0/8975 [00:00<?, ?it/s]C:\Users\sayantan\AppData\Local\Temp\ipykernel_16316\1858854298.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Epoch 1/5 - training: 100%|██████████| 8975/8975 [23:33<00:00,  6.35it/s, avg_loss=1.6010, lr=4.08e-5]



Epoch 1 training finished. avg_loss: 1.5999


Epoch 1 - validating:   0%|          | 0/998 [00:00<?, ?it/s]C:\Users\sayantan\AppData\Local\Temp\ipykernel_16316\1858854298.py:195: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Epoch 1 - validating: 100%|██████████| 998/998 [07:12<00:00,  2.31it/s]


...

Epoch 5/5 - training: 100%|██████████| 8975/8975 [21:55<00:00,  6.82it/s, avg_loss=0.9789, lr=3.18e-8]



Epoch 5 training finished. avg_loss: 0.9789


Epoch 5 - validating: 100%|██████████| 998/998 [06:41<00:00,  2.48it/s]



Validation -- avg_loss: 0.5788, EM: 0.1338, ROUGE-L: 0.6119
 PRED: no polypoid lesions identified
 TRUE: no polypoid lesions identified
----
 PRED: residual polyps present
 TRUE: residual polyps remain
----
 PRED: No text observed
 TRUE: No visible text observed
----
Saved best model to checkpoints_blip2_t5\best_epoch5_rouge0.6119.pt

Training finished. Best ROUGE-L: 0.6118918430635268
Final model saved to: checkpoints_blip2_t5\final_model.pt


: 